# 09. SQL Case When & Scalar Functions: Beginner Guide

### 📝 Universal SQL Execution Order (All Clauses Combined):
```text
┌─ Complete All-in-One SQL Logical Execution Pipeline ─────────────────────────┐
│ 1. WITH (CTEs)       ➔ 2. FROM & JOIN (ON)   ➔ 3. WHERE (Row Filter)         │
│ ➔ 4. GROUP BY        ➔ 5. HAVING (Agg Filter)➔ 6. WINDOW (OVER / Partition)  │
│ ➔ 7. SELECT & CASE   ➔ 8. DISTINCT (Dedup)   ➔ 9. ORDER BY (Sorting)         │
│ ➔ 10. LIMIT / OFFSET (Final Page Slice)                                      │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

### 📌 Overview & Architectural Context
Welcome to **09. SQL Case When & Scalar Functions**. Scalar functions process individual column values row by row to produce transformed scalar outputs. This notebook covers inline conditional branching (`CASE WHEN ... THEN ... ELSE ... END`), string manipulation functions (`UPPER`, `LOWER`, `TRIM`, `SUBSTR`), date parsing and temporal arithmetic, type coercion via `CAST()`, and fallback coalesce functions (`COALESCE`, `NULLIF`).

### 📚 Key Concepts Covered in this Notebook:
- [x] 🔹 Inline Conditional Branching: `CASE WHEN ... THEN ... ELSE ... END`
- [x] 🔹 Fallback Null Handling: `COALESCE(val1, val2, default)`
- [x] 🔹 Division by Zero Protection: `NULLIF(val, 0)`
- [x] 🔹 Explicit Data Type Coercion: `CAST(col AS target_type)`
- [x] 🔍 Scenario: Credit Risk Tier Assignment & Multi-Currency Normalization










In [1]:
# Setup in-memory SQLite relational engine with Native SQL Studio Execution
import sqlite3
import pandas as pd
import os
from IPython import get_ipython
from IPython.core.magic import register_line_cell_magic

conn = sqlite3.connect(':memory:')

def load_table(name, path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, index=False, if_exists='replace')

load_table('transactions', 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv')
load_table('customers', 'data/customers.csv' if os.path.exists('data/customers.csv') else '../data/customers.csv')
load_table('merchants', 'data/merchants.csv' if os.path.exists('data/merchants.csv') else '../data/merchants.csv')
load_table('disputes', 'data/disputes.csv' if os.path.exists('data/disputes.csv') else '../data/disputes.csv')

def _execute_raw_sql(query):
    query = query.strip()
    if query.upper().startswith(('INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'VACUUM', 'ANALYZE', 'BEGIN', 'COMMIT', 'ROLLBACK', 'SAVEPOINT')):
        cur = conn.cursor()
        cur.executescript(query)
        conn.commit()
        return "Query Executed Successfully."
    else:
        return pd.read_sql_query(query, conn)

# Register automatic raw SQL transformer & %%sql magic
ip = get_ipython()
if ip is not None:
    def raw_sql_transformer(lines):
        clean_text = ''.join(lines).strip()
        first_token = clean_text.split()[0].upper() if clean_text.split() else ''
        sql_keywords = {'SELECT', 'WITH', 'INSERT', 'UPDATE', 'DELETE', 'CREATE', 'DROP', 'ALTER', 'EXPLAIN', 'ANALYZE', 'VACUUM', 'BEGIN', 'COMMIT', 'ROLLBACK'}
        if first_token in sql_keywords:
            return [f'_execute_raw_sql("""{clean_text}""")']
        return lines
    
    if raw_sql_transformer not in ip.input_transformers_cleanup:
        ip.input_transformers_cleanup.append(raw_sql_transformer)

@register_line_cell_magic
def sql(line, cell=None):
    return _execute_raw_sql(cell if cell is not None else line)

print("SQL Studio Environment Active! You can now write and run pure SQL queries directly.")


SQL Studio Environment Active! You can now write and run pure SQL queries directly.


### 🔹 Inline Conditional Routing: `CASE WHEN`
- **What it does:** Evaluates a series of boolean conditions in order and returns the corresponding `THEN` result for the first matching predicate.
- **Syntax:** `CASE WHEN cond1 THEN res1 WHEN cond2 THEN res2 ELSE default_res END`
- **Dataset Application & Code Demonstration:** Categorizes transactions into discrete value tiers (Low, Medium, High, VIP).


In [2]:
%%sql
SELECT 
    transaction_id,
    transaction_amount,
    CASE 
        WHEN transaction_amount < 250.00 THEN 'Low Value'
        WHEN transaction_amount BETWEEN 250.00 AND 1000.00 THEN 'Medium Value'
        WHEN transaction_amount BETWEEN 1000.01 AND 1800.00 THEN 'High Value'
        ELSE 'VIP / Whale'
    END AS spend_tier
FROM transactions
WHERE transaction_amount IS NOT NULL
LIMIT 6;


,transaction_id,transaction_amount,spend_tier
0,TX109326,607.78,Medium Value
1,TX106376,1819.11,VIP / Whale
2,TX103301,64.08,Low Value
3,TX110701,1025.73,High Value
4,TX103284,772.74,Medium Value
5,TX104210,198.47,Low Value


### 🔹 Fallback Value Coalescence: `COALESCE()`
- **What it does:** Evaluates its arguments sequentially from left to right and returns the first non-null expression.
- **Syntax:** `COALESCE(expr_1, expr_2, ..., fallback_expr)`
- **Dataset Application & Code Demonstration:** Imputes missing transaction amounts with $0.00.


In [3]:
%%sql
SELECT 
    transaction_id,
    transaction_amount,
    COALESCE(transaction_amount, 0.00) AS safe_transaction_amount
FROM transactions
WHERE transaction_amount IS NULL
LIMIT 5;


,transaction_id,transaction_amount,safe_transaction_amount
0,TX114893,None,0.0
1,TX109183,None,0.0
2,TX113996,None,0.0
3,TX110503,None,0.0
4,TX111012,None,0.0


### 🔹 Division by Zero Safeguard: `NULLIF()`
- **What it does:** Compares two expressions and returns `NULL` if they are equal; otherwise returns the first expression.
- **Syntax:** `NULLIF(expression_1, expression_2)`
- **Dataset Application & Code Demonstration:** Computes transaction ratios safely.


In [4]:
%%sql
SELECT 
    transaction_id,
    transaction_amount,
    account_age_months,
    ROUND(transaction_amount / NULLIF(account_age_months, 0), 2) AS spend_per_account_month
FROM transactions
LIMIT 5;


,transaction_id,transaction_amount,account_age_months,spend_per_account_month
0,TX109326,607.78,8,75.97
1,TX106376,1819.11,28,64.97
2,TX103301,64.08,91,0.70
3,TX110701,1025.73,50,20.51
4,TX103284,772.74,5,154.55


### 🔹 Explicit Type Conversion: `CAST()`
- **What it does:** Converts an expression explicitly from one SQL data type to another (`INTEGER`, `REAL`, `TEXT`).
- **Syntax:** `CAST(expression AS target_type)`
- **Dataset Application & Code Demonstration:** Casts transaction amounts into integer dollars.


In [5]:
%%sql
SELECT 
    transaction_id,
    transaction_amount,
    CAST(transaction_amount AS INTEGER) AS truncated_dollar_amount
FROM transactions
WHERE transaction_amount IS NOT NULL
LIMIT 5;


,transaction_id,transaction_amount,truncated_dollar_amount
0,TX109326,607.78,607
1,TX106376,1819.11,1819
2,TX103301,64.08,64
3,TX110701,1025.73,1025
4,TX103284,772.74,772


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: Complex Customer Risk Scoring with Multi-Factor CASE Logic
- **Objective:** Construct a weighted risk score based on transaction amount, account age, and payment method.
- **Approach:** Combine scalar functions and nested `CASE` evaluations.


In [6]:
%%sql
SELECT 
    customer_id,
    transaction_id,
    transaction_amount,
    account_age_months,
    CASE 
        WHEN is_fraud = 1 THEN 'CRITICAL_RISK'
        WHEN transaction_amount > 1500.00 AND account_age_months < 6 THEN 'HIGH_RISK'
        WHEN transaction_amount > 1000.00 THEN 'MEDIUM_RISK'
        ELSE 'LOW_RISK'
    END AS automated_risk_tier
FROM transactions
LIMIT 6;


,customer_id,transaction_id,transaction_amount,account_age_months,automated_risk_tier
0,C55082,TX109326,607.78,8,LOW_RISK
1,C76616,TX106376,1819.11,28,CRITICAL_RISK
2,C65296,TX103301,64.08,91,LOW_RISK
3,C42098,TX110701,1025.73,50,MEDIUM_RISK
4,C97782,TX103284,772.74,5,LOW_RISK
5,C59823,TX104210,198.47,51,LOW_RISK
